In [1]:
from typing import TypedDict,List
from langgraph.graph import START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
import asyncio
import gzip
import json
from sentence_transformers import SentenceTransformer
import psycopg2
from pgvector.psycopg2 import register_vector
from tqdm import tqdm
import torch
print(torch.backends.mps.is_available())
import time

True


In [2]:
load_dotenv()
GROQ_API_KEY=os.environ.get('GROQ_API_KEY')

In [3]:
conn = psycopg2.connect(
    database=os.environ.get("POSTGRES_DATABASE"),
    user=os.environ.get("POSTGRES_USER"),
    password=os.environ.get("POSTGRES_PASSWORD"),
    host=os.environ.get("HOST"),
    port=os.environ.get("PORT")
)
cursor = conn.cursor()
register_vector(conn)

In [4]:
FINAL_PATH='/Volumes/VV-BACKUP/Book_Recommender/OpenLibData'
FINAL_FILE='clean_books.jsonl'

In [ ]:
def open_and_process_json_file(final_path,final_file):
    total_processed=0
    total_saved=0
    error_count=0
    with open(f'{final_path}/{final_file}', 'w') as out:
        with gzip.open(f'{final_path}/ol_editions.txt.gz',mode='rt',encoding="utf-8") as f:
            for line in f:
                total_processed+=1
                if total_processed % 500000 == 0:
                    print(f"{'-'*23}{total_processed}{'-'*23}")
                line_split=line.split("\t")
                if len(line_split)==5:
                    lib_id=line_split[1]
                    try:
                        json_data=json.loads(line_split[4])
                        if isinstance(json_data.get('description'),dict):
                            json_data['description']=json_data['description']['value']
                        if json_data.get('title') and json_data.get('description') and len(json_data.get('subjects',[])) and json_data.get('authors',[]):
                            data={'ol_id':lib_id,'title':json_data['title'],'author':json_data['authors'],'subjects':json_data['subjects'],'description':json_data['description']}
                            file=json.dumps(data)
                            out.write(file + '\n')
                            total_saved+=1
                            if total_saved% 500000 == 0:
                                print(f"{'-'*23}{total_saved}{'-'*23}")
                    except Exception as e:
                        if error_count%10000==0:
                            print(f"Error: {e}")
                        error_count+=1
                        continue
    print(f"Done processing GZIP FILE \nTotal Processed: {total_processed}\nTotal Saved: {total_saved}\nErrors: {error_count}")
open_and_process_json_file(FINAL_PATH,FINAL_FILE)

-----------------------500000-----------------------
-----------------------1000000-----------------------
-----------------------1500000-----------------------
-----------------------2000000-----------------------
-----------------------2500000-----------------------
-----------------------3000000-----------------------
-----------------------3500000-----------------------
-----------------------4000000-----------------------
-----------------------4500000-----------------------
-----------------------5000000-----------------------
-----------------------5500000-----------------------
-----------------------6000000-----------------------
-----------------------6500000-----------------------
-----------------------7000000-----------------------
-----------------------7500000-----------------------
-----------------------8000000-----------------------
-----------------------8500000-----------------------
-----------------------9000000-----------------------
-----------------------950000

In [ ]:
SENTENCETRANSFORMER=SentenceTransformer("all-MiniLM-L6-v2",device="mps")
def write_embeddings(file_path,json_file):
    record_batch=[]
    text_batch=[]
    with open(f'{file_path}/{json_file}','rt') as f:
        for line in f:
            lines=json.loads(line)
            subjects_str = ", ".join(lines.get('subjects', []))
            json_str=f"title: {lines.get('title')} |subjects: {subjects_str} |description: {(lines.get('description')or '')[:300]}"
            text_batch.append(json_str)
            record_batch.append({"ol_id":lines.get('ol_id'),"title":lines.get('title'),"author":[a.get('key') for a in lines.get('authors', [])],"subjects":lines.get('subjects'),"description":lines.get('description')})
            if len(text_batch)==2048:
                start = time.time()
                vectors=SENTENCETRANSFORMER.encode(text_batch,batch_size=128,show_progress_bar=True,convert_to_tensor=True)
                encode_time = time.time() - start
                start = time.time()
                row=[(record['ol_id'], record['title'], record['author'], record['subjects'], record['description'], vectors[i].cpu().tolist()) 
                    for i, record in enumerate(record_batch)]
                query=f"""
                INSERT INTO {os.environ.get('POSTGRES_SCHEMA')}.book 
                (ol_id,title,author,subjects,description,embedding) VALUES (%s, %s, %s, %s, %s, %s)"""
                cursor.executemany(query,row)
                conn.commit()
                insert_time = time.time() - start
                print(f"Encode: {encode_time:.2f}s | Insert: {insert_time:.2f}s")
                print(f"List has been inserted to book table successfully...")
                text_batch.clear()
                record_batch.clear()
    vectors=SENTENCETRANSFORMER.encode(text_batch,batch_size=128,show_progress_bar=True,convert_to_tensor=True)
    row=[(record['ol_id'], record['title'], record['author'], record['subjects'], record['description'], vectors[i].cpu().tolist()) 
         for i, record in enumerate(record_batch)]
    query=f"""INSERT INTO {os.environ.get('POSTGRES_SCHEMA')}.book (ol_id,title,author,subjects,description,embedding) VALUES (%s, %s, %s, %s, %s, %s)"""
    cursor.executemany(query,row)
    conn.commit()
    conn.close()
    print("Final Rows inserted...")        
write_embeddings(FINAL_PATH,FINAL_FILE)

In [ ]:
class error_data_schema(TypedDict):
    user_input:str
    failed_step:List[str]
    error_message:List[str]
    error_type:List[str]
    context:List[str]
    search_results:str
    user_response:str

In [ ]:
def user_inputs_log(user_input,state=error_data_schema)->error_data_schema:
    state.user_input=user_input
    return state